<a id="inizio"></a>
# Approfondimenti 2 · Soluzioni

⏱ ~40 min &nbsp;·&nbsp; Dati: `../Dati/load_total_north_hourly_2024.xlsx`

**In questo notebook impariamo a**

- misurare durate con `Timedelta` e dare un fuso orario alle date con `tz_localize` e `tz_convert`
- navigare una serie lunga con range slider e range selector
- esportare un grafico interattivo in HTML

Le sezioni sulle date usano `df_15min`, la serie costruita nel notebook 08. Contiene una potenza e
una temperatura inventate, con un valore ogni 15 minuti per tre giorni, e i buchi di acquisizione
sono già riempiti con l'interpolazione lineare. Le due celle seguenti importano le librerie e
ricostruiscono la serie, in modo che questo notebook si possa eseguire da solo.

In [ ]:
import numpy as np
np.random.seed(42)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
intervallo = pd.date_range("2025-03-01 00:00", periods=3 * 24 * 4, freq="15min")
ore = intervallo.hour + intervallo.minute / 60
power_kw = 220 + 60 * np.sin(2 * np.pi * (ore / 24)) + np.random.normal(0, 8, size=len(intervallo))
temp_c = 12 + 5 * np.sin(2 * np.pi * ((ore - 6) / 24)) + np.random.normal(0, 0.7, size=len(intervallo))

misure = pd.DataFrame({"timestamp": intervallo, "power_kw": power_kw.round(1), "temp_c": temp_c.round(1)})
misure = misure.drop(np.random.choice(misure.index, size=18, replace=False))  # i buchi

df_15min = misure.set_index("timestamp").asfreq("15min").interpolate(method="linear")
df_15min.head()

<a id="sez-1"></a>
## 1. Differenze tra date

Sottraendo due timestamp si ottiene un `Timedelta`, cioè una durata. Le durate servono a due scopi:
controllare la regolarità di una serie, guardando la distanza tra ogni istante e il successivo, e
misurare l'intervallo complessivo coperto dai dati. Nella prima cella calcoliamo con `diff()` la
differenza tra ogni timestamp e il precedente; il primo valore è `NaT`, perché prima della prima
riga non c'è nulla da sottrarre.

In [ ]:
# differenze tra timestamp consecutivi
delta_consecutivi = df_15min.index.to_series().diff()

delta_consecutivi.head(10)

La durata totale della serie è la differenza tra l'ultimo e il primo timestamp dell'indice. Un
`Timedelta` si converte in un numero con `total_seconds()`, e dividendo i secondi per 3600 otteniamo
le ore coperte dal dataset.

In [ ]:
# intervallo totale coperto
delta_totale = df_15min.index.max() - df_15min.index.min()
delta_totale

In [ ]:
# durata in ore (float)
delta_totale.total_seconds() / 3600

<a id="sez-2"></a>
## 2. Fuso orario

Un timestamp può essere **naive**, cioè composto solo da data e ora senza indicazione del fuso,
oppure **timezone-aware**, quando porta con sé un fuso orario come `Europe/Rome`. Capita spesso di
ricevere orari locali naive, che vanno interpretati nel fuso giusto prima di confrontarli con dati di
altre fonti. La difficoltà principale è l'ora legale: nel giorno del cambio di primavera un'ora non
esiste, mentre in quello d'autunno un'ora si ripete due volte.

La documentazione di pandas dedica una sezione alla [gestione dei fusi orari](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#time-zone-handling).

<div style="background:rgba(217,83,79,0.12); border-left:4px solid #d9534f; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**⚠️ Attenzione**

Quando si localizza un indice che attraversa un cambio dell'ora legale, alcuni timestamp risultano
ambigui o inesistenti e `tz_localize` solleva un errore. In questi casi si indica come trattarli con
i parametri `ambiguous=` e `nonexistent=`. Se i dati arrivano già in UTC, conviene in genere
mantenerli in UTC e convertirli all'ora locale solo al momento di mostrarli.

</div>

Con `tz_localize` dichiariamo in quale fuso vanno letti i timestamp naive dell'indice, senza
cambiare l'ora scritta. Con `tz_convert` riscriviamo poi gli stessi istanti nell'ora di un altro
fuso: il 1° marzo l'Italia è un'ora avanti rispetto a UTC, quindi la mezzanotte italiana diventa le
23 del giorno prima. Su un indice con fuso orario la selezione per intervallo con `loc` funziona come
prima, e le stringhe vengono interpretate nel fuso dell'indice.

In [ ]:
df_fuso = df_15min.copy()

# interpretiamo i timestamp come orari locali italiani
df_fuso.index = df_fuso.index.tz_localize("Europe/Rome")

# conversione a UTC (utile per sistemi e confronti)
df_utc = df_fuso.tz_convert("UTC")

df_fuso.index[:3], df_utc.index[:3]

In [ ]:
df_fuso.loc["2025-03-01 06:00":"2025-03-01 12:00"].head()

<a id="sez-3"></a>
## 3. Range slider e range selector

Per navigare una serie storica lunga Plotly offre due controlli sull'asse x: il **range slider**,
una barra sotto il grafico con cui si sceglie l'intervallo da mostrare, e il **range selector**, una
fila di pulsanti che saltano a intervalli predefiniti. Per usarli passiamo a `graph_objects` e
seguiamo lo schema degli [esempi ufficiali](https://plotly.com/python/range-slider/), nel quale
prima si costruiscono le tracce e poi si aggiunge il controllo nel `layout`.

Come dati usiamo `px.data.stocks()`, lo stesso dataset del notebook 09. Contiene una colonna con la
data, `date`, e una colonna per ciascuna azienda con il prezzo dell'azione indicizzato, cioè diviso
per il valore iniziale, in modo che tutte le serie partano da 1. La colonna `date` è memorizzata come
testo, quindi la convertiamo subito in datetime.

In [ ]:
df = px.data.stocks().copy()
df["date"] = pd.to_datetime(df["date"])
df.head()

Con `go.Figure()` creiamo una figura vuota e con `add_trace` aggiungiamo una linea per ciascuna
delle tre aziende scelte. Il nome della colonna diventa il nome della traccia, che compare nella
legenda.

In [ ]:
aziende = ["AAPL", "AMZN", "MSFT"]

fig = go.Figure()
for c in aziende:
    fig.add_trace(go.Scatter(x=list(df["date"]), y=list(df[c]), name=c))

I pulsanti del range selector si descrivono con una lista di dizionari. Ogni pulsante indica, con
`count` e `step`, di quanto tornare indietro rispetto all'ultima data, e con `label` l'etichetta da
mostrare; il pulsante con `step="all"` riporta il grafico all'intervallo completo. Nell'esempio
definiamo quattro pulsanti: un mese, sei mesi, un anno e l'intero periodo.

In [ ]:
bottoni = [
    dict(count=1, label="1m", step="month", stepmode="backward"),
    dict(count=6, label="6m", step="month", stepmode="backward"),
    dict(count=1, label="1y", step="year", stepmode="backward"),
    dict(step="all"),
]

Infine passiamo i pulsanti a `update_layout`, dentro la configurazione dell'asse x, e rendiamo
visibile lo slider con `rangeslider=dict(visible=True)`. Nel grafico i pulsanti compaiono sopra
l'area del disegno e lo slider sotto l'asse; trascinando i bordi dello slider si sceglie un
intervallo qualsiasi.

In [ ]:
# range slider + range selector (schema ufficiale)
fig.update_layout(
    title_text="Stocks (indicizzato), range slider",
    height=400,
    width=700,
    xaxis=dict(
        rangeselector=dict(buttons=bottoni, bgcolor="lightgray", font=dict(size=10)),
        rangeslider=dict(visible=True),
        type="date",
    ),
    margin=dict(t=80, b=80),
)
fig.show()

<a id="sez-4"></a>
## 4. Esportare in HTML

Per condividere un grafico interattivo con chi non usa Jupyter, la strada più semplice è salvarlo
come file HTML, che si apre con qualsiasi browser e conserva zoom, legenda e controlli. Il metodo
`write_html` scrive il file nella cartella del notebook; con `include_plotlyjs="cdn"` il file resta
leggero, perché la libreria Plotly viene scaricata da internet al momento dell'apertura. I dettagli
sono nella [documentazione](https://plotly.com/python/interactive-html-export/).

In [ ]:
# salva l'ultima figura (fig) in HTML, nella cartella del notebook: si apre con un browser
# con include_plotlyjs="cdn" la libreria Plotly si scarica all'apertura: file leggero, ma serve la rete
fig.write_html("plot_time_series.html", include_plotlyjs="cdn")

<a id="sez-5"></a>
## 5. Esercizi

Gli esercizi usano due oggetti dei notebook precedenti. Il primo è `carico`, il carico Terna della
zona Nord nel 2024, ordinato per data e senza i duplicati dovuti all'ora legale come nel notebook 08;
il secondo è `fig_carico`, il grafico di gennaio costruito nell'esercizio 9.1. La cella seguente li
ricostruisce entrambi.

In [ ]:
carico = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
carico = carico.sort_values("Date")
carico = carico.drop_duplicates(subset="Date")

gennaio = carico[carico["Date"].dt.month == 1]
fig_carico = px.line(gennaio, x="Date", y="Total Load [MW]", title="Carico Nord, gennaio 2024")
fig_carico.show()

<a id="es-a2-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio A2.1 · Il range slider sul carico di gennaio**

Il grafico di gennaio contiene quasi 3000 quarti d'ora, troppi per distinguere i singoli giorni
a colpo d'occhio. Con un range slider e due pulsanti possiamo passare rapidamente dalla vista di
una settimana a quella del mese intero.

**Cosa fare.**

1. Attiva il range slider sotto l'asse x di `fig_carico`.
2. Aggiungi un range selector con due pulsanti: il primo, con etichetta `1w`, torna indietro di una settimana (`count=7`, `step="day"`), mentre il secondo, `all`, mostra tutto il mese.

Se il codice è corretto, sotto il grafico compare lo slider e sopra compaiono i due pulsanti.

*Suggerimento:* su una figura creata con `px` lo slider si attiva con `fig_carico.update_xaxes(rangeslider_visible=True)`, e il range selector si aggiunge nella stessa chiamata con `rangeselector=dict(buttons=...)`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione A2.1**

</div>

In [ ]:
bottoni_carico = [
    dict(count=7, label="1w", step="day", stepmode="backward"),
    dict(step="all"),
]
fig_carico.update_xaxes(rangeslider_visible=True, rangeselector=dict(buttons=bottoni_carico))
fig_carico.show()

In [ ]:
from corso import verifica
verifica("A2.1")

<a id="es-a2-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio A2.2 · Il grafico del carico in HTML**

Dobbiamo mandare il grafico del carico di gennaio, `fig_carico`, a un collega che non ha Python
installato ma vuole comunque poterlo esplorare con lo zoom.

**Cosa fare.**

Salva `fig_carico` nel file `carico_gennaio.html` con `include_plotlyjs="cdn"`, poi aprilo con un
doppio clic dalla cartella del notebook. Il file che ottieni è leggero, sotto 1 MB, e nel browser
il grafico resta interattivo.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione A2.2**

Senza `include_plotlyjs="cdn"` il file contiene al suo interno l'intera libreria Plotly, che pesa qualche megabyte, e in compenso si apre anche senza connessione. Con l'opzione `cdn` il file contiene solo i dati e la descrizione del grafico, e la libreria viene scaricata all'apertura.

</div>

In [ ]:
fig_carico.write_html("carico_gennaio.html", include_plotlyjs="cdn")

In [ ]:
from corso import verifica
verifica("A2.2")

---

**Fine.**

<sub>Approfondimenti: se la classe è avanti · Giornata 2 · Aula Base</sub>